# 02 - Data Cleaning

Objectif : garder les colonnes utiles, convertir les nombres et exporter un fichier nettoye.

In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

# Fichier source : donnees brutes a nettoyer
raw_path = project_root / 'data' / 'raw' / 'Building_Energy_Benchmarking_Data,_2015-Present_20260704.csv'

# Fichier de sortie : donnees nettoyees creees par ce notebook
cleaned_path = project_root / 'data' / 'cleaned' / 'buildings_clean.csv'

## Selectionner les colonnes utiles

In [2]:
df_raw = pd.read_csv(raw_path, low_memory=False)
raw_row_count = len(df_raw)

columns_to_keep = {
    'OSEBuildingID': 'building_id',
    'DataYear': 'data_year',
    'BuildingName': 'building_name',
    'BuildingType': 'building_type',
    'Neighborhood': 'neighborhood',
    'YearBuilt': 'year_built',
    'NumberofFloors': 'number_of_floors',
    'PropertyGFATotal': 'property_gfa_total',
    'ENERGYSTARScore': 'energy_star_score',
    'SiteEUI(kBtu/sf)': 'site_eui',
    'SiteEnergyUse(kBtu)': 'site_energy_use',
    'Electricity(kWh)': 'electricity_kwh',
    'NaturalGas(therms)': 'natural_gas_therms',
    'TotalGHGEmissions': 'total_ghg_emissions',
    'GHGEmissionsIntensity': 'ghg_emissions_intensity',
    'ComplianceStatus': 'compliance_status'
}

df = df_raw[list(columns_to_keep.keys())].rename(columns=columns_to_keep).copy()
df.head()

,building_id,data_year,building_name,building_type,neighborhood,year_built,number_of_floors,property_gfa_total,energy_star_score,site_eui,site_energy_use,electricity_kwh,natural_gas_therms,total_ghg_emissions,ghg_emissions_intensity,compliance_status
0,1,2024,MAYFLOWER PARK HOTEL,NonResidential,DOWNTOWN,1927,12,"88,434",59.0,61.70000076,"7,113,958","1,045,040","15,986",263.3,2.98,Not Compliant
1,2,2024,PARAMOUNT HOTEL,NonResidential,DOWNTOWN,1996,11,"103,566",85.0,71.5,"6,330,664","787,838","36,426",208.6,2.36,Compliant
2,3,2024,WESTIN HOTEL (Parent Building),NonResidential,DOWNTOWN,1969,41,"956,110",71.0,81.69999695,"67,613,264","11,279,080","58,726","2,418.2",3.18,Compliant
3,5,2024,HOTEL MAX,NonResidential,DOWNTOWN,1926,10,"61,320",50.0,86,"5,273,981","796,976","11,648",190.1,3.1,Compliant
4,8,2024,WARWICK SEATTLE HOTEL,NonResidential,DOWNTOWN,1980,18,"175,580",87.0,97.09999847,"11,983,959","1,349,025","73,811",417.9,3.68,Compliant


## Nettoyer les types et les valeurs

In [3]:
numeric_columns = [
    'building_id', 'data_year', 'year_built', 'number_of_floors',
    'property_gfa_total', 'energy_star_score', 'site_eui', 'site_energy_use',
    'electricity_kwh', 'natural_gas_therms', 'total_ghg_emissions',
    'ghg_emissions_intensity'
]

numeric_updates = {}
for column in numeric_columns:
    values = df[column].astype('string').str.replace(',', '', regex=False).str.strip()
    values = values.mask(values.isin(['', 'NA', 'nan', '<NA>']))
    numeric_updates[column] = pd.to_numeric(values, errors='coerce')

df = df.assign(**numeric_updates)

# Noms et quartiers sont en MAJUSCULES dans la source : on les met en forme.
# Les types de batiment et statuts sont deja lisibles (LR, MR, HR, COS, SPS...) :
# on ne les passe pas en .title(), seule la casse de 'NonResidential' est harmonisee.
df = df.assign(
    building_name=df['building_name'].fillna('Unknown').astype(str).str.strip().str.title(),
    neighborhood=df['neighborhood'].fillna('Unknown').astype(str).str.strip().str.title(),
    building_type=(
        df['building_type'].fillna('Unknown').astype(str).str.strip()
        .replace({'NonResidential': 'Nonresidential'})
    ),
    compliance_status=df['compliance_status'].fillna('Unknown').astype(str).str.strip(),
)

df = df.drop_duplicates().copy()

# On retire les lignes sans valeurs essentielles pour les KPI.
critical_columns = [
    'building_id', 'data_year', 'year_built', 'number_of_floors',
    'property_gfa_total', 'site_eui', 'site_energy_use', 'electricity_kwh',
    'total_ghg_emissions', 'ghg_emissions_intensity'
]
df = df.dropna(subset=critical_columns).copy()

# Regles simples de coherence metier.
df = df[df['property_gfa_total'] > 0].copy()
df = df[df['site_eui'] > 0].copy()
df = df[df['site_energy_use'] > 0].copy()
df = df[df['number_of_floors'] > 0].copy()
df = df[df['year_built'] <= df['data_year']].copy()

# Valeurs aberrantes : une intensite energetique hors de [1 ; 1000] kBtu/sf
# n'est pas realiste pour un batiment (mediane ~37, 99,9e centile ~740).
# Ces lignes sont des erreurs de declaration probables (ex. un immeuble
# de bureaux declare a 5,8 milliards de kBtu) qui faussent totaux et moyennes.
EUI_MIN, EUI_MAX = 1, 1000
eui_outliers = ~df['site_eui'].between(EUI_MIN, EUI_MAX)
outlier_row_count = int(eui_outliers.sum())
df = df[~eui_outliers].copy()

# Le score ENERGY STAR manquant est conserve comme valeur manquante : le
# remplacer par la mediane classerait artificiellement ces batiments en 'Bon'.
# Les moyennes (pandas, SQL, Tableau) ignorent automatiquement ces valeurs.
df = df.assign(
    energy_star_score_was_missing=df['energy_star_score'].isna().astype(int),
    natural_gas_was_missing=df['natural_gas_therms'].isna().astype(int)
)

# Si le gaz naturel manque, on considere 0 (batiment sans gaz) pour les agregations.
df = df.assign(natural_gas_therms=df['natural_gas_therms'].fillna(0))

integer_columns = ['building_id', 'data_year', 'year_built', 'number_of_floors']
df[integer_columns] = df[integer_columns].astype('int64')

print(f'Lignes aberrantes retirees (site_eui hors [{EUI_MIN} ; {EUI_MAX}]) : {outlier_row_count}')
df.head()

Lignes aberrantes retirees (site_eui hors [1 ; 1000]) : 29


,building_id,data_year,building_name,building_type,neighborhood,year_built,number_of_floors,property_gfa_total,energy_star_score,site_eui,site_energy_use,electricity_kwh,natural_gas_therms,total_ghg_emissions,ghg_emissions_intensity,compliance_status,energy_star_score_was_missing,natural_gas_was_missing
0,1,2024,Mayflower Park Hotel,Nonresidential,Downtown,1927,12,88434,59.0,61.700001,7113958,1045040,15986,263.3,2.98,Not Compliant,0,0
1,2,2024,Paramount Hotel,Nonresidential,Downtown,1996,11,103566,85.0,71.5,6330664,787838,36426,208.6,2.36,Compliant,0,0
2,3,2024,Westin Hotel (Parent Building),Nonresidential,Downtown,1969,41,956110,71.0,81.699997,67613264,11279080,58726,2418.2,3.18,Compliant,0,0
3,5,2024,Hotel Max,Nonresidential,Downtown,1926,10,61320,50.0,86.0,5273981,796976,11648,190.1,3.1,Compliant,0,0
4,8,2024,Warwick Seattle Hotel,Nonresidential,Downtown,1980,18,175580,87.0,97.099998,11983959,1349025,73811,417.9,3.68,Compliant,0,0


## Controle qualite

In [4]:
quality_check = pd.DataFrame({
    'column_name': df.columns,
    'missing_values': df.isna().sum().values,
    'missing_rate': (df.isna().mean().values * 100).round(2)
})

cleaning_summary = pd.DataFrame({
    'metric': [
        'Lignes brutes',
        'Lignes apres nettoyage',
        'Lignes retirees',
        'Part des lignes retirees (%)',
        'dont valeurs aberrantes (site_eui)',
        'Batiments distincts',
        'Scores ENERGY STAR non renseignes',
        'Valeurs gaz naturel completees (0)'
    ],
    'value': [
        raw_row_count,
        len(df),
        raw_row_count - len(df),
        round((raw_row_count - len(df)) / raw_row_count * 100, 2),
        outlier_row_count,
        df['building_id'].nunique(),
        int(df['energy_star_score_was_missing'].sum()),
        int(df['natural_gas_was_missing'].sum())
    ]
})

display(cleaning_summary)
quality_check

,metric,value
0,Lignes brutes,34699.00
1,Lignes apres nettoyage,33239.00
2,Lignes retirees,1460.00
3,Part des lignes retirees (%),4.21
4,dont valeurs aberrantes (site_eui),29.00
5,Batiments distincts,3806.00
6,Scores ENERGY STAR non renseignes,7908.00
7,Valeurs gaz naturel completees (0),12144.00


,column_name,missing_values,missing_rate
0,building_id,0,0.00
1,data_year,0,0.00
2,building_name,0,0.00
3,building_type,0,0.00
4,neighborhood,0,0.00
5,year_built,0,0.00
6,number_of_floors,0,0.00
7,property_gfa_total,0,0.00
8,energy_star_score,7908,23.79
9,site_eui,0,0.00


### Conclusion du controle qualite

Les lignes sans valeurs essentielles pour les KPI ont ete retirees, ainsi que les cas incoherents : surface nulle, consommation nulle, nombre d'etages egal a 0, annee de construction posterieure a l'annee de reporting, et intensite energetique (`site_eui`) hors de l'intervalle realiste [1 ; 1000] kBtu/sf.

Chaque ligne correspond a **un batiment pour une annee de declaration** : le nombre de lignes n'est donc pas le nombre de batiments (voir `Batiments distincts`).

Les scores ENERGY STAR manquants sont **laisses vides** : les remplacer par la mediane les aurait classes artificiellement en categorie `Bon`. La seule valeur manquante restante du fichier est donc `energy_star_score`. Les valeurs manquantes de gaz naturel ont ete remplacees par 0, car elles correspondent le plus souvent a des batiments qui n'utilisent pas cette energie.

Deux colonnes indicatrices restent disponibles : `energy_star_score_was_missing` et `natural_gas_was_missing`.

## Exporter les donnees nettoyees

In [5]:
cleaned_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(cleaned_path, index=False)
print(f'Fichier exporte : {cleaned_path}')
print(f'Lignes exportees : {len(df):,}')

Fichier exporte : C:\Users\djami\Desktop\Projects\Energy-Performance-Dashboard\data\cleaned\buildings_clean.csv
Lignes exportees : 33,239
